# Multi-block encoding visualization (Lexical Delay)

Compare **semantic / phon / acoustic** marginals and **full_perm_semantic** (controlled).

Works with **partial** Slurm results — inventory shows coverage.

Style: `docs/PLOTTING_STYLE.md` / vizpub fig2–fig4.


In [ ]:
import importlib
import os
from pathlib import Path

os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")
os.environ.setdefault("VTK_DEFAULT_RENDER_WINDOW_OFFSCREEN", "1")
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

import rootutils


def _find_project_root(start: Path | None = None) -> Path:
    starts: list[Path] = []
    if start is not None:
        starts.append(Path(start))
    starts.append(Path.cwd())
    starts.append(Path.cwd() / "insula-semantic")
    starts.append(Path("/hpc/group/coganlab/nanlinshi/insula-semantic"))
    seen: set[Path] = set()
    for raw in starts:
        base = raw.resolve()
        if base in seen:
            continue
        seen.add(base)
        for candidate in (base, *base.parents):
            if (candidate / ".project-root").exists():
                return candidate
    raise FileNotFoundError("Project root not found.")


PROJECT_ROOT = _find_project_root()
rootutils.setup_root(PROJECT_ROOT, indicator=".project-root", pythonpath=True, cwd=True)

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pyvista as pv
import seaborn as sns
from IPython.display import display
from mne.viz import Brain
from scipy.spatial.distance import cdist

import src.semantic.load_encoding_results as load_encoding_results

importlib.reload(load_encoding_results)
from src.semantic.load_encoding_results import (
    DEFAULT_RESULTS_DIR,
    MULTI_MODELS,
    load_encoding_long_multi,
)


In [ ]:
# vizpub style
cm = 1 / 2.54
plt.rcParams["svg.fonttype"] = "none"
fontdict = dict(fontsize=7)
fontsize = 7

red = "#A9373B"
blue = "#2369BD"
orange = "#CC8963"
green = "#009944"

MODEL_COLORS = {
    "semantic": red,
    "phon": blue,
    "acoustic": orange,
    "full_perm_semantic": green,
}
MODEL_LABELS = {
    "semantic": "sem marginal",
    "phon": "phon marginal",
    "acoustic": "acous marginal",
    "full_perm_semantic": "sem controlled",
}

recon_dir = "/cwork/ns458/ECoG_Recon/"


def style_axes(ax, offset: float = 1, trim: bool = True) -> None:
    ax.tick_params(labelsize=fontsize, width=0.75, length=2, which="both")
    plt.setp(ax.spines.values(), linewidth=0.75)
    sns.despine(ax=ax, offset=offset, trim=trim)


def setup_brain_offscreen() -> None:
    if getattr(setup_brain_offscreen, "_done", False):
        return
    import warnings

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        try:
            pv.start_xvfb()
        except Exception:
            pass
    mne.viz.set_3d_backend("notebook")
    setup_brain_offscreen._done = True


setup_brain_offscreen()


## 1. Load multi-block results (partial OK)


In [ ]:
RESULTS_DIR = DEFAULT_RESULTS_DIR
MODELS = list(MULTI_MODELS)
PHASES = ["Stimulus", "Delay", "Go", "Response"]
DESCRIPTIONS = ["Decision", "Repeat"]
EXPECTED_SUBJECTS = 52

TIME_WINDOWS = {
    "Stimulus": (0.0, 0.5),
    "Delay": (0.0, 0.5),
    "Go": (0.0, 0.5),
    "Response": (-0.5, 0.5),
}

channels_all, long_all = load_encoding_long_multi(
    results_dir=RESULTS_DIR,
    models=MODELS,
)
print(f"Loaded channel rows: {len(channels_all):,}  long rows: {len(long_all):,}")
if channels_all.empty:
    raise RuntimeError("No multi-block H5 files found yet.")

inventory = (
    channels_all.groupby(["model", "description", "phase"])["subject"]
    .nunique()
    .unstack("phase")
    .reindex(columns=PHASES)
    .fillna(0)
    .astype(int)
)
print("Subjects with H5 (by model × description × phase):")
display(inventory)

n_files = int(channels_all.groupby(["subject", "model", "phase", "description"]).ngroups)
print(
    f"Condition files loaded: {n_files} / "
    f"{EXPECTED_SUBJECTS * len(MODELS) * len(PHASES) * len(DESCRIPTIONS)}"
)

channels_all["significant"] = channels_all["ch_sig_any"].fillna(False).astype(bool)
long_all["significant"] = long_all["significant"].fillna(False).astype(bool)

peak_keys = ["subject", "channel", "phase", "description", "model"]
peak_sig = (
    long_all.loc[long_all["significant"] == True, peak_keys + ["abs_r"]]
    .groupby(peak_keys, as_index=False)["abs_r"]
    .max()
    .rename(columns={"abs_r": "peak_sig_abs_r"})
)
channels_all = channels_all.merge(peak_sig, on=peak_keys, how="left")
channels_all["effect"] = channels_all["peak_sig_abs_r"].fillna(0.0)


def spatial_sig_in_window(phase: str, description: str, model: str) -> pd.DataFrame:
    tmin, tmax = TIME_WINDOWS[phase]
    sub_long = long_all[
        (long_all["phase"] == phase)
        & (long_all["description"] == description)
        & (long_all["model"] == model)
        & (long_all["time"] >= tmin)
        & (long_all["time"] <= tmax)
        & (long_all["significant"] == True)
    ]
    if sub_long.empty:
        return pd.DataFrame()
    keys = ["subject", "channel", "phase", "description", "model"]
    hit = sub_long[keys].drop_duplicates()
    return channels_all.merge(hit, on=keys, how="inner")


## 2. Significance summary (time-window electrodes)

Cluster-significant channels inside each phase window, by model.


In [ ]:
rows = []
for model in MODELS:
    for desc in DESCRIPTIONS:
        for phase in PHASES:
            sp = spatial_sig_in_window(phase, desc, model)
            n_sub = int(
                channels_all.loc[
                    (channels_all.model == model)
                    & (channels_all.description == desc)
                    & (channels_all.phase == phase),
                    "subject",
                ].nunique()
            )
            rows.append(
                {
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "description": desc,
                    "phase": phase,
                    "n_sig_ch": 0 if sp.empty else int(len(sp)),
                    "n_subjects": n_sub,
                    "n_sig_sub": 0 if sp.empty else int(sp["subject"].nunique()),
                }
            )
window_counts = pd.DataFrame(rows)
display(window_counts)

fig, axes = plt.subplots(1, 2, figsize=(14 * cm, 4.5 * cm), sharey=True)
x = np.arange(len(MODELS))
for ax, desc in zip(axes, DESCRIPTIONS):
    sub = (
        window_counts[(window_counts["phase"] == "Delay") & (window_counts["description"] == desc)]
        .set_index("model")
        .reindex(MODELS)
    )
    ax.bar(
        x,
        sub["n_sig_ch"],
        color=[MODEL_COLORS[m] for m in MODELS],
        width=0.7,
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODELS], rotation=25, ha="right")
    ax.set_title(f"Delay · {desc}", **fontdict)
    style_axes(ax)
axes[0].set_ylabel("Sig channels (in window)", **fontdict)
fig.suptitle("Cluster-sig electrodes in Delay window (partial cohort)", fontsize=fontsize, y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(MODELS), figsize=(18 * cm, 4 * cm), sharey=True)
for ax, model in zip(axes, MODELS):
    sub = (
        window_counts[(window_counts["model"] == model) & (window_counts["description"] == "Decision")]
        .set_index("phase")
        .reindex(PHASES)
    )
    ax.bar(
        range(len(PHASES)),
        sub["n_sig_ch"],
        color=MODEL_COLORS[model],
        width=0.65,
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_xticks(range(len(PHASES)))
    ax.set_xticklabels(PHASES, rotation=25, ha="right")
    ax.set_title(MODEL_LABELS[model], **fontdict)
    style_axes(ax)
axes[0].set_ylabel("Sig channels", **fontdict)
fig.suptitle("Decision: sig channels by phase (partial cohort)", fontsize=fontsize, y=1.02)
plt.tight_layout()
plt.show()


## 3. ROI breakdown (Delay window)


In [ ]:
roi_rows = []
for model in MODELS:
    for desc in DESCRIPTIONS:
        sp = spatial_sig_in_window("Delay", desc, model)
        if sp.empty:
            continue
        for roi, n in sp["roi"].value_counts().items():
            roi_rows.append(
                {
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "description": desc,
                    "roi": roi,
                    "n": int(n),
                }
            )
roi_win = pd.DataFrame(roi_rows)
if roi_win.empty:
    print("No significant electrodes in Delay window yet.")
else:
    pivot = roi_win.groupby(["model_label", "description", "roi"], as_index=False)["n"].sum()
    top = (
        pivot.groupby(["model_label", "roi"], as_index=False)["n"]
        .sum()
        .sort_values("n", ascending=False)
    )
    print("Top ROIs in Delay window (summed Decision+Repeat):")
    display(top.head(24))

    focus = {"AIC", "PIC", "STGa", "STGp", "HG", "IFG", "MFG", "PrG", "PoG"}
    plot_df = pivot[(pivot["description"] == "Decision") & (pivot["roi"].isin(focus))]
    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(12 * cm, 5 * cm))
        sns.barplot(
            data=plot_df,
            x="roi",
            y="n",
            hue="model_label",
            palette={MODEL_LABELS[m]: MODEL_COLORS[m] for m in MODELS},
            ax=ax,
        )
        ax.set_xlabel("ROI", **fontdict)
        ax.set_ylabel("Sig channels", **fontdict)
        ax.set_title("Delay · Decision · selected ROIs", **fontdict)
        ax.legend(frameon=False, fontsize=fontsize, title=None)
        style_axes(ax)
        plt.tight_layout()
        plt.show()


## 4. Inflated KDE — Delay × Decision (4 models)

Gaussian KDE on `x_t,y_t,z_t` (same as `semantic_encode_viz.ipynb`). Rows = models; columns = lh / rh.


In [ ]:
COORD_COLS = ["x_t", "y_t", "z_t"]
KDE_BANDWIDTH = 8.0
KDE_MAX_DISTANCE = 12.0
KDE_DENSITY_OPACITY = 0.75
KDE_N_COLORS = 1000

lh_pial_coords, _ = mne.read_surface(f"{recon_dir}/cvs_avg35_inMNI152/surf/lh.pial")
rh_pial_coords, _ = mne.read_surface(f"{recon_dir}/cvs_avg35_inMNI152/surf/rh.pial")
lh_infl_coords, _ = mne.read_surface(f"{recon_dir}/cvs_avg35_inMNI152/surf/lh.inflated")
rh_infl_coords, _ = mne.read_surface(f"{recon_dir}/cvs_avg35_inMNI152/surf/rh.inflated")


def compute_density_unified(surface_coords, electrode_coords, effect_values, bandwidth=15, max_distance=30):
    if len(electrode_coords) == 0:
        return np.zeros(len(surface_coords)), np.zeros(len(surface_coords), dtype=bool)
    distances = cdist(surface_coords, electrode_coords)
    min_distances = distances.min(axis=1)
    mask = min_distances < max_distance
    kernel_values = np.exp(-(distances ** 2) / (2 * bandwidth ** 2))
    density = np.dot(kernel_values, np.abs(effect_values))
    density[~mask] = 0
    return density, mask


def make_inflated_brain(hemi):
    return Brain(
        "cvs_avg35_inMNI152",
        subjects_dir=recon_dir,
        surf="inflated",
        hemi=hemi,
        background="white",
        show=False,
        cortex=(0.95, 0.95, 0.95),
        alpha=0.85,
        size=(800, 800),
    )


def _brain_screenshot(brain, mode="rgb"):
    setup_brain_offscreen()
    brain._renderer.plotter.render()
    return brain.screenshot(mode=mode)


def _kde_colormap(base_rgb, n_colors=KDE_N_COLORS, opacity=KDE_DENSITY_OPACITY):
    colors_list = [[1.0, 1.0, 1.0, 1.0]]
    base = np.asarray(mcolors.to_rgb(base_rgb), dtype=float)
    for i in range(1, n_colors):
        intensity = i / (n_colors - 1)
        color = (1.0 - intensity) * np.array([1.0, 1.0, 1.0]) + intensity * base
        colors_list.append(list(color) + [opacity])
    return mcolors.ListedColormap(colors_list)


def _hemi_density(spatial_df, hemi):
    pial = lh_pial_coords if hemi == "lh" else rh_pial_coords
    hemi_code = "L" if hemi == "lh" else "R"
    sub = spatial_df[
        (spatial_df["hemi"] == hemi_code) & (~spatial_df[COORD_COLS].isna().any(axis=1))
    ]
    if sub.empty:
        return np.zeros(len(pial)), np.zeros(len(pial), dtype=bool), 0
    dens, mask = compute_density_unified(
        pial,
        sub[COORD_COLS].values,
        sub["effect"].values,
        bandwidth=KDE_BANDWIDTH,
        max_distance=KDE_MAX_DISTANCE,
    )
    return dens, mask, len(sub)


def _render_inflated_kde_panel(density, dens_mask, vmin, vmax, hemi, cmap):
    setup_brain_offscreen()
    brain = make_inflated_brain(hemi)
    n_vertices = len(lh_infl_coords if hemi == "lh" else rh_infl_coords)
    data_to_plot = np.zeros(n_vertices)
    has_data = dens_mask & (density > 0)
    if has_data.any():
        norm = np.clip((density[has_data] - vmin) / (vmax - vmin + 1e-12), 0.0, 1.0)
        data_to_plot[has_data] = 1.0 + norm * (KDE_N_COLORS - 2)
    brain.add_data(
        data_to_plot,
        hemi=hemi,
        colormap=cmap,
        alpha=1.0,
        colorbar=False,
        fmin=0,
        fmax=KDE_N_COLORS - 1,
    )
    try:
        brain.add_annotation("aparc.a2009s", borders=True, color="gray", alpha=0.35)
    except Exception:
        pass
    brain.show_view(view="lateral", distance=400)
    img = _brain_screenshot(brain)
    try:
        brain.close()
    except Exception:
        pass
    return img


PHASE_PLOT = "Delay"
DESC_PLOT = "Decision"
fig, axes = plt.subplots(len(MODELS), 2, figsize=(12 * cm, 3.2 * cm * len(MODELS)))
for row, model in enumerate(MODELS):
    sp = spatial_sig_in_window(PHASE_PLOT, DESC_PLOT, model)
    cmap = _kde_colormap(MODEL_COLORS[model])
    dens_l, mask_l, n_l = _hemi_density(sp, "lh") if not sp.empty else (None, None, 0)
    dens_r, mask_r, n_r = _hemi_density(sp, "rh") if not sp.empty else (None, None, 0)
    vals = []
    if dens_l is not None and mask_l is not None and mask_l.any():
        vals.append(dens_l[mask_l])
    if dens_r is not None and mask_r is not None and mask_r.any():
        vals.append(dens_r[mask_r])
    if vals:
        allv = np.concatenate(vals)
        vmin, vmax = float(np.percentile(allv, 5)), float(np.percentile(allv, 95))
        if vmax <= vmin:
            vmax = vmin + 1e-6
    else:
        vmin, vmax = 0.0, 1.0
    for col, hemi, dens, dmask, n_el in [
        (0, "lh", dens_l, mask_l, n_l),
        (1, "rh", dens_r, mask_r, n_r),
    ]:
        ax = axes[row, col]
        if dens is None or n_el == 0:
            ax.imshow(np.ones((20, 30, 3)))
            ax.set_title(f"{MODEL_LABELS[model]} · {hemi} (n=0)", **fontdict)
        else:
            img = _render_inflated_kde_panel(dens, dmask, vmin, vmax, hemi, cmap)
            ax.imshow(img)
            ax.set_title(f"{MODEL_LABELS[model]} · {hemi} (n={n_el})", **fontdict)
        ax.axis("off")
fig.suptitle(
    f"Inflated KDE · {PHASE_PLOT} × {DESC_PLOT} (partial cohort)",
    fontsize=fontsize,
    y=1.01,
)
plt.tight_layout()
plt.show()


## Notes

- Re-run after Slurm finishes for full 52 × 8 coverage.
- Primary claim map: **full_perm_semantic** (green).
- Marginals diagnose phon / acoustic / bare GloVe structure.
